# RAMSES Movie Map Viewer

Notebook for inspecting RAMSES `movie*` map outputs like `dens_00424.map`, `vx_00424.map`, etc.

It uses `utils/py/miniramses.py::rd_map` for reading and provides:
- single-frame plotting
- interactive frame browsing
- optional inline animation
- optional `amr2vid.py` command generation for MP4 output


In [24]:
%matplotlib inline

from pathlib import Path
import re
import sys
import subprocess

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

sys.path.append("../utils/py")

import miniramses as ram
#!/usr/bin/env python3
import os
import numpy as np
import matplotlib.pyplot as plt
import miniramses as ram

# -----------------------------
# Matplotlib config
# -----------------------------
plt.rcParams["font.family"] = "serif"
plt.rcParams["figure.dpi"] = 150
plt.rcParams.update({
    "font.size": 14,
    "legend.fontsize": 12,
})
plt.rcParams['animation.embed_limit'] = 100.0 

In [25]:
# Edit these for your run
#movie_dir = Path("/home/jl4415/mini-ramses/coeur_fmm/movie1/")  # e.g. Path("/scratch/.../movie1")
# Edit these for your run
movie_dir = Path("/home/jl4415/mini-ramses/halo_test/halo_mg/movie1/") 
variable = "dens"                 # "dens", "vx", "vy", "vz", "temp", ...
use_log = True
cmap = "inferno"
vmin = None
vmax = None


In [26]:
def discover_map_files(movie_dir: Path, variable: str):
    movie_dir = Path(movie_dir)
    pattern = re.compile(rf"^{re.escape(variable)}_(\d+)\.map$")
    records = []
    for p in movie_dir.glob(f"{variable}_*.map"):
        m = pattern.match(p.name)
        if m:
            records.append((int(m.group(1)), p))
    records.sort(key=lambda x: x[0])
    return records

map_records = discover_map_files(movie_dir, variable)
if not map_records:
    raise FileNotFoundError(f"No files matching {variable}_*.map in {movie_dir}")

frame_numbers = [n for n, _ in map_records]
map_files = [p for _, p in map_records]

print(f"Found {len(map_files)} files for '{variable}' in {movie_dir}")
print(f"First frame number: {frame_numbers[0]:05d}")
print(f"Last frame number:  {frame_numbers[-1]:05d}")
print(f"Example file:       {map_files[0]}")


FileNotFoundError: No files matching dens_*.map in /home/jl4415/mini-ramses/halo_test/halo_mg/movie1

In [27]:
def read_frame(idx: int):
    m = ram.rd_map(str(map_files[idx]))
    data = np.array(m.data, copy=False)
    if use_log:
        data = np.log10(np.maximum(data, 1e-30))
    return m, data

def plot_frame(idx: int, figsize=(6, 6), save=False, out_dir=None):
    idx = int(np.clip(idx, 0, len(map_files) - 1))
    frame_no = frame_numbers[idx]
    m, data = read_frame(idx)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(data.T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar_label = f"log10({variable})" if use_log else variable
    plt.colorbar(im, ax=ax, label=cbar_label)
    ax.set_title(f"{variable}_{frame_no:05d}.map   t={m.time:.6e}")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    plt.tight_layout()

    if save:
        if out_dir is None:
            out_dir = movie_dir / "notebook_frames"
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        out = out_dir / f"{variable}_{frame_no:05d}.png"
        fig.savefig(out, dpi=140)
        print(f"Saved {out}")

    return fig, ax

# Example: plot first frame
plot_frame(0)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/home/jl4415/mini-ramses/halo_test/halo_fmm/movie1/dens_00001.map'

In [28]:
# Optional: inline animation preview (for quick inspection)
def make_animation(step=1, max_frames=120, interval_ms=80):
    idxs = list(range(0, len(map_files), max(1, int(step))))[:max_frames]
    if not idxs:
        raise ValueError("No frames selected")

    m0, d0 = read_frame(idxs[0])
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(d0.T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    cbar_label = f"log10({variable})" if use_log else variable
    plt.colorbar(im, ax=ax, label=cbar_label)
    title = ax.set_title(f"{variable}_{frame_numbers[idxs[0]]:05d}.map   t={m0.time:.6e}")

    def update(j):
        idx = idxs[j]
        m, d = read_frame(idx)
        im.set_data(d.T)
        title.set_text(f"{variable}_{frame_numbers[idx]:05d}.map   t={m.time:.6e}")
        return im, title

    ani = FuncAnimation(fig, update, frames=len(idxs), interval=interval_ms, blit=False)
    plt.close(fig)
    return ani

ani = make_animation(step=5, max_frames=1000, interval_ms=70)
HTML(ani.to_jshtml())


FileNotFoundError: [Errno 2] No such file or directory: '/home/jl4415/mini-ramses/halo_test/halo_fmm/movie1/dens_00001.map'

In [7]:
# Optional: build MP4 with amr2vid.py (same logic as CLI tool)
start = frame_numbers[0]
end = frame_numbers[-1]
output_mp4 = movie_dir / f"{variable}.mp4"

cmd = [
    sys.executable,
    str("../utils/py/amr2vid.py"),
    str(start),
    str(end),
    "--mode", "map",
    "--variable", variable,
    "--movie-dir", str(movie_dir),
    "--col", cmap,
    "--output", str(output_mp4),
    "--keep-frames",
]
if use_log:
    cmd.append("--log")

print("Command:")
print(" ".join(cmd))

# Uncomment to execute
subprocess.run(cmd, check=True)


Command:
/usr/licensed/anaconda3/2021.11/bin/python ../utils/py/amr2vid.py 1 928 --mode map --variable dens --movie-dir /home/jl4415/mini-ramses/coeur_fmm/movie1 --col inferno --output /home/jl4415/mini-ramses/coeur_fmm/movie1/dens.mp4 --keep-frames --log
Frame directory: /home/jl4415/mini-ramses/analyze/frames
Mode: map
Output movie: /home/jl4415/mini-ramses/coeur_fmm/movie1/dens.mp4
Looking for movie directories in: /home/jl4415/mini-ramses/coeur_fmm/movie1
Map mode: processing movie *.map files
Current working directory: /home/jl4415/mini-ramses/analyze
Using movie directory: /home/jl4415/mini-ramses/coeur_fmm/movie1
Processing variable: dens
Found 928 map files
      1: dens_00001.map
      2: dens_00002.map
      3: dens_00003.map
      4: dens_00004.map
      5: dens_00005.map
      6: dens_00006.map
      7: dens_00007.map
      8: dens_00008.map
      9: dens_00009.map
     10: dens_00010.map
     11: dens_00011.map
     12: dens_00012.map
     13: dens_00013.map
     14: dens_

  Output 00879 -> Frame 00879
  Output 00880 -> Frame 00880
  Output 00881 -> Frame 00881
  Output 00882 -> Frame 00882
  Output 00883 -> Frame 00883
  Output 00884 -> Frame 00884
  Output 00885 -> Frame 00885
  Output 00886 -> Frame 00886
  Output 00887 -> Frame 00887
  Output 00888 -> Frame 00888
  Output 00889 -> Frame 00889
  Output 00890 -> Frame 00890
  Output 00891 -> Frame 00891
  Output 00892 -> Frame 00892
  Output 00893 -> Frame 00893
  Output 00894 -> Frame 00894
  Output 00895 -> Frame 00895
  Output 00896 -> Frame 00896
  Output 00897 -> Frame 00897
  Output 00898 -> Frame 00898
  Output 00899 -> Frame 00899
  Output 00900 -> Frame 00900
  Output 00901 -> Frame 00901
  Output 00902 -> Frame 00902
  Output 00903 -> Frame 00903
  Output 00904 -> Frame 00904
  Output 00905 -> Frame 00905
  Output 00906 -> Frame 00906
  Output 00907 -> Frame 00907
  Output 00908 -> Frame 00908
  Output 00909 -> Frame 00909
  Output 00910 -> Frame 00910
  Output 00911 -> Frame 00911
  Output 0

Generated frame 00034 from dens_00034.map (output 00034)
Running command: python /home/jl4415/mini-ramses/utils/py/map2img.py /home/jl4415/mini-ramses/coeur_fmm/movie1/dens_00035.map --no-display --log --col inferno --out /home/jl4415/mini-ramses/analyze/frames/frame_00035.png
Working directory: /home/jl4415/mini-ramses/analyze
Map file: /home/jl4415/mini-ramses/coeur_fmm/movie1/dens_00035.map
Generated frame from dens_00035.map
Generated frame 00035 from dens_00035.map (output 00035)
Running command: python /home/jl4415/mini-ramses/utils/py/map2img.py /home/jl4415/mini-ramses/coeur_fmm/movie1/dens_00036.map --no-display --log --col inferno --out /home/jl4415/mini-ramses/analyze/frames/frame_00036.png
Working directory: /home/jl4415/mini-ramses/analyze
Map file: /home/jl4415/mini-ramses/coeur_fmm/movie1/dens_00036.map
Generated frame from dens_00036.map
Generated frame 00036 from dens_00036.map (output 00036)
Running command: python /home/jl4415/mini-ramses/utils/py/map2img.py /home/jl

KeyboardInterrupt: 